# Basic CF example using a book recommender system data.
### Data from https://www.kaggle.com/datasets/rxsraghavagrawal/book-recommender-system?resource=download

In [ ]:
# Import library

import tensorflow as tf
print(tf.__version__)

2.19.0


In [ ]:
# Import data

from google.colab import files
uploaded = files.upload()

Saving BX-Book-Ratings.csv to BX-Book-Ratings.csv
Saving BX-Books.csv to BX-Books.csv
Saving BX-Users.csv to BX-Users.csv


In [ ]:
import pandas as pd
# Load the ratings data (this file contains User-ID, ISBN, and Book-Rating)
ratings = pd.read_csv("BX-Book-Ratings.csv", sep=";", encoding="latin-1")

# Preview basic structure
print("Rows, Cols:", ratings.shape)
print("Columns:", list(ratings.columns))

# Preview the first 10 rows
ratings.head(10)

Rows, Cols: (1149780, 3)
Columns: ['User-ID', 'ISBN', 'Book-Rating']


,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6
5,276733,2080674722,0
6,276736,3257224281,8
7,276737,0600570967,6
8,276744,038550120X,7
9,276745,342310538,10


In [ ]:
# Count total rows
total_rows = ratings.shape[0]

# Count how many ratings are 0 (0 ratings mean that that the user didn't rate that book)
num_zero = (ratings["Book-Rating"] == 0).sum()

# Count how many are non-zero
num_nonzero = (ratings["Book-Rating"] > 0).sum()

print("Total rows:", total_rows)
print("Zero ratings:", num_zero)
print("Non-zero ratings:", num_nonzero)

print("Percent zero:", round(num_zero / total_rows * 100, 2), "%")
print("Percent non-zero:", round(num_nonzero / total_rows * 100, 2), "%")

Total rows: 1149780
Zero ratings: 716109
Non-zero ratings: 433671
Percent zero: 62.28 %
Percent non-zero: 37.72 %


In [ ]:
# Remove rows where Book-Rating is 0
ratings = ratings[ratings["Book-Rating"] > 0].copy()

print("New shape after removing zeros:", ratings.shape)

# Quick check of rating distribution
ratings["Book-Rating"].describe()

New shape after removing zeros: (433671, 3)


,Book-Rating
count,433671.000000
mean,7.601066
std,1.843798
min,1.000000
25%,7.000000
50%,8.000000
75%,9.000000
max,10.000000


In [ ]:
# Convert raw User-ID and ISBN values into consecutive integer indices.
# TensorFlow embedding layers require integer inputs from 0 to N-1, not raw IDs or strings.

import numpy as np

# Extract unique users and books from the dataset
user_ids = ratings["User-ID"].unique()
book_isbns = ratings["ISBN"].unique()

# Create dictionaries that map each raw ID to a new integer index
user_to_index = {u: i for i, u in enumerate(user_ids)}
isbn_to_index = {isbn: i for i, isbn in enumerate(book_isbns)}

# Add new columns with these integer indices
ratings["user_index"] = ratings["User-ID"].map(user_to_index).astype("int32")
ratings["book_index"] = ratings["ISBN"].map(isbn_to_index).astype("int32")

# Show how many unique users and books exist
print("Number of users:", len(user_ids))
print("Number of books:", len(book_isbns))

# Preview the transformation
ratings[["User-ID", "ISBN", "Book-Rating", "user_index", "book_index"]].head()

Number of users: 77805
Number of books: 185973


,User-ID,ISBN,Book-Rating,user_index,book_index
1,276726,0155061224,5,0,0
3,276729,052165615X,3,1,1
4,276729,0521795028,6,1,2
6,276736,3257224281,8,2,3
7,276737,0600570967,6,3,4


In [ ]:
# Convert the prepared indices into TensorFlow tensors and create an 80/20 train-test split.
# We shuffle the data first to make the split random.

import tensorflow as tf

# Convert columns to tensors
user_tensor = tf.convert_to_tensor(ratings["user_index"].values, dtype=tf.int32)
book_tensor = tf.convert_to_tensor(ratings["book_index"].values, dtype=tf.int32)
rating_tensor = tf.convert_to_tensor(ratings["Book-Rating"].values, dtype=tf.float32)

# Combine user and book indices into one feature tensor
X = tf.stack([user_tensor, book_tensor], axis=1)
y = rating_tensor

# Shuffle all samples
n = tf.shape(X)[0]
indices = tf.random.shuffle(tf.range(n), seed=42)
X = tf.gather(X, indices)
y = tf.gather(y, indices)

# 80 percent train, 20 percent test
train_size = tf.cast(tf.cast(n, tf.float32) * 0.8, tf.int32)

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

print("Total samples:", int(n.numpy()))
print("Training samples:", int(tf.shape(X_train)[0].numpy()))
print("Test samples:", int(tf.shape(X_test)[0].numpy()))

Total samples: 433671
Training samples: 346936
Test samples: 86735


In [ ]:
# Build a matrix factorization model using TensorFlow embeddings.
# The model learns a vector for each user and each book, then predicts ratings using dot product plus biases.

import tensorflow as tf

num_users = int(ratings["user_index"].nunique())
num_books = int(ratings["book_index"].nunique())
embedding_dim = 32

# Model inputs
user_in = tf.keras.Input(shape=(1,), dtype=tf.int32, name="user_index")
book_in = tf.keras.Input(shape=(1,), dtype=tf.int32, name="book_index")

# Embeddings turn user and book indices into latent vectors
user_vec = tf.keras.layers.Flatten()(
    tf.keras.layers.Embedding(num_users, embedding_dim, name="user_embedding")(user_in)
)
book_vec = tf.keras.layers.Flatten()(
    tf.keras.layers.Embedding(num_books, embedding_dim, name="book_embedding")(book_in)
)

# Dot product models how well a user and book match
dot = tf.keras.layers.Dot(axes=1)([user_vec, book_vec])

# Bias terms capture user rating scale and book popularity effects
user_bias = tf.keras.layers.Flatten()(
    tf.keras.layers.Embedding(num_users, 1, name="user_bias")(user_in)
)
book_bias = tf.keras.layers.Flatten()(
    tf.keras.layers.Embedding(num_books, 1, name="book_bias")(book_in)
)

# Final prediction
pred = tf.keras.layers.Add()([dot, user_bias, book_bias])

model = tf.keras.Model(inputs=[user_in, book_in], outputs=pred)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="mse",
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_index          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ book_index          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │  2,489,760 │ user_index[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ book_embedding      │ (None, 1, 32)     │  5,951,136 │ book_index[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ book_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_bias           │ (None, 1, 1)      │     77,805 │ user_index[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ book_bias           │ (None, 1, 1)      │    185,973 │ book_index[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 1)         │          0 │ user_bias[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 1)         │          0 │ book_bias[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1)         │          0 │ dot[0][0],        │
│                     │                   │            │ flatten_2[0][0],  │
│                     │                   │            │ flatten_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,704,674 (33.21 MB)

 Trainable params: 8,704,674 (33.21 MB)

 Non-trainable params: 0 (0.00 B)

The model summary represents a neural matrix factorization architecture implemented using embedding layers. It learns a 32 dimensional latent vector for every user and every book, where these vectors capture abstract preference patterns and item characteristics inferred from rating behavior. The predicted rating is computed as the dot product between the user and book embeddings, which measures their compatibility, and is then adjusted by user specific and book specific bias terms to account for individual rating tendencies and overall item popularity. The total number of parameters shown corresponds to all learned user embeddings, book embeddings, and their associated bias values.

In [ ]:
# Create efficient TensorFlow datasets and train the model.
# Data is batched for faster computation and trained for 5 epochs.

batch_size = 4096

train_ds = tf.data.Dataset.from_tensor_slices(
    ((X_train[:, 0], X_train[:, 1]), y_train)
).shuffle(100_000).batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices(
    ((X_test[:, 0], X_test[:, 1]), y_test)
).batch(batch_size).prefetch(tf.data.AUTOTUNE)

history = model.fit(train_ds, validation_data=test_ds, epochs=5)

Epoch 1/5
85/85 ━━━━━━━━━━━━━━━━━━━━ 15s 133ms/step - loss: 57.7613 - mae: 7.3650 - val_loss: 48.3915 - val_mae: 6.6581
Epoch 2/5
85/85 ━━━━━━━━━━━━━━━━━━━━ 16s 93ms/step - loss: 41.5939 - mae: 6.0773 - val_loss: 34.5559 - val_mae: 5.3407
Epoch 3/5
85/85 ━━━━━━━━━━━━━━━━━━━━ 8s 92ms/step - loss: 17.9994 - mae: 3.5639 - val_loss: 28.9303 - val_mae: 4.7096
Epoch 4/5
85/85 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - loss: 6.2052 - mae: 1.8487 - val_loss: 27.7607 - val_mae: 4.5694
Epoch 5/5
85/85 ━━━━━━━━━━━━━━━━━━━━ 8s 90ms/step - loss: 2.0808 - mae: 0.9980 - val_loss: 27.5906 - val_mae: 4.5483


The training MAE decreased to about 1.00, while the validation MAE stabilized around 4.55. This indicates the model fits the training data very well but shows some overfitting on unseen data, which is common in collaborative filtering with large embedding tables.

In [ ]:
# For demonstration purposes, we randomly select 3 users who have rated at least 3 books.
# We then display the top 3 books each user rated highest and compare them to the model’s
# top 3 recommendations for that same user. This comparison helps us visually evaluate
# whether the collaborative filtering model is capturing the user’s taste.
#
# Note: Some users in the dataset have very few ratings (sometimes fewer than 3).
# This highlights a known limitation of collaborative filtering called the "cold start problem."
# When a user has provided very few ratings, the model has limited information to learn
# their preferences, which can lead to weaker or more generic recommendations.


import numpy as np
import pandas as pd
from IPython.display import display, HTML

def user_top_rated(user_index, n=3):
    user_rows = ratings[ratings["user_index"] == int(user_index)].copy()
    user_rows = user_rows.merge(books[["ISBN", "Book-Title", "Book-Author"]], on="ISBN", how="left")

    user_rows["Book-Title"] = user_rows["Book-Title"].fillna("Unknown title")
    user_rows["Book-Author"] = user_rows["Book-Author"].fillna("Unknown author")

    user_rows = user_rows.sort_values("Book-Rating", ascending=False)

    out = user_rows[["Book-Title", "Book-Author", "Book-Rating"]].head(n).reset_index(drop=True)
    out.index = np.arange(1, len(out) + 1)
    out.index.name = "Rank"
    return out

def model_recs(user_index, top_n=3, candidate_pool=50000):
    num_books = int(ratings["book_index"].nunique())
    all_books = np.arange(num_books, dtype=np.int32)

    already_rated = rated_books_by_user.get(int(user_index), set())
    unseen_books = np.array([b for b in all_books if b not in already_rated], dtype=np.int32)

    if unseen_books.size > candidate_pool:
        candidate_books = np.random.choice(unseen_books, size=candidate_pool, replace=False)
    else:
        candidate_books = unseen_books

    user_vec = np.full(shape=(candidate_books.shape[0],), fill_value=int(user_index), dtype=np.int32)
    preds = model.predict([user_vec, candidate_books], batch_size=8192, verbose=0).reshape(-1)

    top_idx = np.argsort(preds)[::-1][:top_n]
    top_books = candidate_books[top_idx]

    recs = pd.DataFrame({"book_index": top_books.astype(int)})
    recs["ISBN"] = recs["book_index"].map(book_index_to_isbn)

    recs = recs.merge(books[["ISBN", "Book-Title", "Book-Author"]], on="ISBN", how="left")

    recs["Book-Title"] = recs["Book-Title"].fillna("Unknown title")
    recs["Book-Author"] = recs["Book-Author"].fillna("Unknown author")

    out = recs[["Book-Title", "Book-Author"]].head(top_n).reset_index(drop=True)
    out.index = np.arange(1, len(out) + 1)
    out.index.name = "Rank"
    return out

def style_table(df):
    return (
        df.style
        .set_table_styles([
            {"selector": "th", "props": "font-size:14px; text-align:center; background:#f5f5f5;"},
            {"selector": "td", "props": "font-size:13px; padding:6px;"},
            {"selector": "caption", "props": "caption-side: top; font-weight:700; font-size:15px; padding:6px 0;"},
        ])
        .set_properties(**{"text-align": "left"})
        .hide(axis="index", level=0)  # hides Rank index header spacing in some colab themes
    )

def show_user_report(user_index, n_top_rated=3, n_recs=3):
    header = f"""
    <div style="margin:18px 0 8px 0; padding:10px 12px; border-radius:10px; background:#fafafa; border:1px solid #e6e6e6;">
      <div style="font-size:16px; font-weight:700;">User index {user_index}</div>
      <div style="font-size:13px; color:#555;">Ratings in data: {int(user_rating_counts.loc[user_index])}</div>
    </div>
    """
    display(HTML(header))

    top_df = user_top_rated(user_index, n=n_top_rated)
    rec_df = model_recs(user_index, top_n=n_recs)

    styled_top = (
        top_df.style
        .set_caption("User top rated books")
        .set_table_styles([
            {"selector": "th", "props": "font-size:14px; text-align:left; background:#f3f7ff;"},
            {"selector": "td", "props": "font-size:13px; padding:6px;"},
            {"selector": "caption", "props": "caption-side: top; font-weight:700; font-size:15px; padding:6px 0;"},
        ])
        .set_properties(**{"text-align": "left"})
    )

    styled_rec = (
        rec_df.style
        .set_caption("Model recommendations")
        .set_table_styles([
            {"selector": "th", "props": "font-size:14px; text-align:left; background:#f6fff3;"},
            {"selector": "td", "props": "font-size:13px; padding:6px;"},
            {"selector": "caption", "props": "caption-side: top; font-weight:700; font-size:15px; padding:6px 0;"},
        ])
        .set_properties(**{"text-align": "left"})
    )

    display(styled_top)
    display(styled_rec)

random_users = np.random.choice(eligible_users, size=3, replace=False)
for u in random_users:
    show_user_report(int(u), n_top_rated=3, n_recs=3)

NameError: name 'eligible_users' is not defined